# Football Analytics — esplorazione dei dati (M4)Questo notebook serve a **capire**, non a produrre. Niente logica applicativaqui dentro: cio' che serve davvero vive in `src/football_analytics/`.Legge il magazzino costruito da `scripts/build_dataset.py` — 43.849 tiri da1.753 partite — e cerca cio' che non era ovvio prima di guardarlo.Alla fine, le **tre domande** a cui la dashboard rispondera'.> Dati forniti da StatsBomb Open Data.

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio

from football_analytics import config

pio.templates.default = "plotly_white"
VERDE = "#0F6E56"

tiri = pd.read_parquet(config.percorso_tabella("shots"))
partite = pd.read_parquet(config.percorso_tabella("matches"))
giocatori = pd.read_parquet(config.percorso_tabella("player_stats"))

# I rigori finali sono tiri, ma non sono gioco: restano fuori da ogni analisi.
gioco = tiri[~tiri["rigori_finali"]].copy()
gioco["distanza"] = np.hypot(120 - gioco["x"], 40 - gioco["y"])

print(f"tiri {len(gioco):,}   partite {len(partite):,}   giocatori {len(giocatori):,}")

tiri 43,659   partite 1,753   giocatori 4,810


---## 1. Quanto vale un tiroLa prima cosa da guardare e' la distribuzione dell'xG, perche' decide tutto ilresto: la metrica di valutazione del modello, il modo di leggere i grafici,persino la scelta di quali tiri mostrare in una mappa.

In [2]:
q = gioco["xg_statsbomb"].quantile([0.1, 0.25, 0.5, 0.75, 0.9, 0.99])
print(f"media   {gioco['xg_statsbomb'].mean():.4f}")
print(f"mediana {gioco['xg_statsbomb'].median():.4f}")
print("\nquantili:")
for k, v in q.items():
    print(f"  {k:>5.0%}  {v:.3f}")

sotto = gioco["xg_statsbomb"] < 0.05
print(f"\ntiri sotto 0,05 di xG:            {sotto.mean():.1%}")
print(f"quota dei gol che producono:      {gioco[sotto]['gol'].sum() / gioco['gol'].sum():.1%}")

media   0.0991
mediana 0.0509

quantili:
    10%  0.014
    25%  0.027
    50%  0.051
    75%  0.102
    90%  0.233
    99%  0.784

tiri sotto 0,05 di xG:            49.3%
quota dei gol che producono:      11.7%


In [3]:
# Pre-aggregato di proposito: passare a Plotly 43.659 valori grezzi
# significherebbe incorporarli tutti nell'output del notebook, quasi un
# megabyte per un grafico che ne mostra sessanta barre.
conteggi, bordi = np.histogram(gioco["xg_statsbomb"], bins=60, range=(0, 1))
centri = (bordi[:-1] + bordi[1:]) / 2

fig = px.bar(
    x=centri, y=conteggi,
    labels={"x": "xG del tiro", "y": "numero di tiri"},
    title="La distribuzione dell'xG e' fortemente asimmetrica",
    color_discrete_sequence=[VERDE],
)
fig.update_layout(height=380, bargap=0.02, showlegend=False)
fig.show()

**Osservazione 1.** La mediana e' **0,051**: meta' dei tiri ha meno di unaprobabilita' su venti di finire in gol. La media, 0,099, e' quasi il doppiodella mediana — la coda a destra tira su il valore centrale.Eppure quei tiri quasi disperati non sono rumore: producono **l'11,7 % ditutti i gol**. Sono tanti, e qualcosa entra.**Conseguenza per M5:** con una classe positiva intorno al 10 %, l'accuratezzae' inutilizzabile — un modello che risponde sempre «non e' gol» arriva al 90 %.Servono Brier score, log loss e curva di calibrazione.

---## 2. Da dove si tira, e da dove si segna

In [4]:
fasce = pd.cut(gioco["distanza"], [0, 6, 11, 16, 22, 30, 150],
               labels=["0-6 m", "6-11 m", "11-16 m", "16-22 m", "22-30 m", "30+ m"])
per_distanza = gioco.groupby(fasce, observed=True).agg(
    tiri=("gol", "size"), gol=("gol", "sum"), conversione=("gol", "mean"),
    xg_medio=("xg_statsbomb", "mean"),
)
per_distanza["quota_tiri"] = per_distanza["tiri"] / len(gioco)
per_distanza["quota_gol"] = per_distanza["gol"] / gioco["gol"].sum()
per_distanza.round(3)

,tiri,gol,conversione,xg_medio,quota_tiri,quota_gol
distanza,,,,,,
0-6 m,1196,523,0.437,0.433,0.027,0.118
6-11 m,7090,1299,0.183,0.174,0.162,0.292
11-16 m,8845,1459,0.165,0.158,0.203,0.328
16-22 m,9480,690,0.073,0.076,0.217,0.155
22-30 m,11963,386,0.032,0.033,0.274,0.087
30+ m,5085,88,0.017,0.015,0.116,0.020


In [5]:
lungo = per_distanza[["quota_tiri", "quota_gol"]].reset_index().melt(
    id_vars="distanza", var_name="misura", value_name="quota",
)
lungo["misura"] = lungo["misura"].map({"quota_tiri": "quota dei tiri", "quota_gol": "quota dei gol"})
fig = px.bar(
    lungo, x="distanza", y="quota", color="misura", barmode="group",
    title="Si tira da lontano, si segna da vicino",
    labels={"quota": "quota sul totale", "distanza": ""},
    color_discrete_sequence=[VERDE, "#5DCAA5"],
)
fig.update_layout(height=380, yaxis_tickformat=".0%")
fig.show()

**Osservazione 2.** La distanza mediana di tiro e' **19 metri**, ben oltre illimite dell'area. I due grafici raccontano lo scarto:- entro **6 metri** si concentra il **2,7 %** dei tiri e l'**11,8 %** dei gol;- oltre **22 metri** c'e' il **39 %** dei tiri e il **10,7 %** dei gol.Quasi due tiri su cinque partono da una zona che rende un gol ogni trentatentativi. Non e' irrazionalita': un tiro da fuori costa poco e a volteproduce un rimpallo. Ma spiega perche' una classifica per numero di tiri dicapoco, e perche' la mappa dei tiri debba dimensionare i cerchi sull'xG.

---## 3. Quanto vale sapere dove sono i difensoriE' la domanda centrale del progetto. Il fotogramma di ogni tiro contiene laposizione di tutti i giocatori inquadrati: qui si guarda solo **quantiavversari** ci sono, che e' l'approssimazione piu' grezza possibile.

In [6]:
azione = gioco[(gioco["tipo"] == "Open Play") & gioco["ha_fotogramma"]].copy()
fasce_av = pd.cut(azione["avversari_fotogramma"], [-1, 2, 3, 4, 5, 6, 7, 40],
                  labels=["<=2", "3", "4", "5", "6", "7", "8+"])
per_avversari = azione.groupby(fasce_av, observed=True).agg(
    tiri=("gol", "size"), conversione=("gol", "mean"),
    xg_medio=("xg_statsbomb", "mean"), distanza_media=("distanza", "mean"),
)
per_avversari.round(3)

,tiri,conversione,xg_medio,distanza_media
avversari_fotogramma,,,,
<=2,38,0.421,0.402,15.899000
3,306,0.288,0.268,16.385000
4,893,0.231,0.215,16.207001
5,1725,0.192,0.174,16.362000
6,2943,0.168,0.141,17.253000
7,5388,0.120,0.114,18.250999
8+,29886,0.073,0.075,19.683001


In [7]:
fig = px.bar(
    per_avversari.reset_index(), x="avversari_fotogramma", y="conversione",
    title="Piu' avversari fra pallone e porta, meno gol — a distanza quasi costante",
    labels={"conversione": "gol per tiro", "avversari_fotogramma": "avversari inquadrati"},
    color_discrete_sequence=[VERDE], text_auto=".1%",
)
fig.update_layout(height=380, yaxis_tickformat=".0%")
fig.show()

**Osservazione 3.** La conversione passa dal **38,9 %** con due avversariinquadrati al **7,2 %** con otto o piu'. Un fattore **cinque**.Il dettaglio che rende l'osservazione solida e' la colonna della distanza: fra3 e 7 avversari resta fra i 16 e i 18 metri, quasi costante. Non e' che conpochi difensori si tira da piu' vicino — **si tira dalla stessa distanza e sisegna molto di piu'**.Questo e' il segnale che il modello con le variabili spaziali dovra' catturare,e la ragione per cui il confronto con il modello base ha senso.

---## 4. La trappola dei rigoriQui c'e' l'osservazione piu' importante del notebook, e non riguarda il calcioma **come sono raccolti i dati**.

In [8]:
rigori = gioco[gioco["tipo"] == "Penalty"]
riepilogo = rigori.groupby("ha_fotogramma").agg(
    rigori=("gol", "size"), gol=("gol", "sum"), conversione=("gol", "mean"),
)
riepilogo.round(3)

,rigori,gol,conversione
ha_fotogramma,,,
False,426,349,0.819
True,54,6,0.111


In [9]:
print("esiti dei rigori, con e senza fotogramma:")
pd.crosstab(rigori["ha_fotogramma"], rigori["esito"])

esiti dei rigori, con e senza fotogramma:


esito,Goal,Off T,Post,Saved,Saved to Post,Wayward
ha_fotogramma,,,,,,
False,349,0,2,73,2,0
True,6,16,12,17,1,2


**Osservazione 4 — e va letta due volte.**| | Rigori | Conversione || --- | ---: | ---: || **senza** fotogramma | 426 | **81,9 %** || **con** fotogramma | 54 | **11,1 %** |Il fotogramma di un rigore contiene **solo il portiere**, e StatsBomb loallega quasi esclusivamente quando il rigore **non entra**: serve a registrarela posizione del portiere per analizzare la parata.**La presenza del dato dipende dal risultato.** E' distorsione da selezione, ede' il tipo di difetto che un modello impara volentieri: usando`ha_fotogramma` come variabile, imparerebbe «fotogramma presente, quindisbagliato» — che non e' calcio, e' un artefatto di raccolta.Due conseguenze per M5:1. i rigori restano **fuori dal modello**, che e' comunque la prassi: hanno xG   fisso e non dipendono dalla posizione dei difensori;2. `ha_fotogramma` non deve **mai** essere una variabile.E la verifica che salva il progetto:

In [10]:
for tipo in ("Open Play", "Free Kick"):
    sub = gioco[gioco["tipo"] == tipo]
    print(f"{tipo:<12} {sub['ha_fotogramma'].mean():.1%} dei tiri ha il fotogramma  ({len(sub):,} tiri)")

Open Play    100.0% dei tiri ha il fotogramma  (41,179 tiri)
Free Kick    100.0% dei tiri ha il fotogramma  (1,987 tiri)


**Sui tiri su azione la copertura e' del 100 %.** Nessuna selezione, nessunadistorsione: il confronto fra modello base e modello spaziale si puo' fare sututti i 41.179 tiri su azione senza che la presenza del dato dica qualcosasull'esito.L'1 % di tiri senza fotogramma nell'intero magazzino sono, praticamente tutti,rigori.

---## 5. Le quattro leghe, stessa stagione

In [11]:
campionati = partite[partite["gruppo"] == "campionato"]
per_lega = campionati.groupby("competizione", observed=True).apply(
    lambda x: pd.Series({
        "partite": len(x),
        "gol_partita": (x["gol_casa"] + x["gol_ospite"]).mean(),
        "tiri_partita": (x["tiri_casa"] + x["tiri_ospite"]).mean(),
        "xg_partita": (x["xg_casa"] + x["xg_ospite"]).mean(),
        "gol_su_xg": (x["gol_casa"] + x["gol_ospite"]).sum() / (x["xg_casa"] + x["xg_ospite"]).sum(),
    }),
    include_groups=False,
)
per_lega.round(3)

,partite,gol_partita,tiri_partita,xg_partita,gol_su_xg
competizione,,,,,
la_liga_2015_16,380.0,2.745,24.126,2.578,1.065
ligue1_2015_16,377.0,2.517,23.379,2.321,1.085
premier_2015_16,380.0,2.700,26.074,2.556,1.056
serie_a_2015_16,380.0,2.576,26.311,2.423,1.063


**Osservazione 5.** Le quattro leghe si somigliano piu' di quanto il raccontocomune suggerisca. I gol per partita vanno da **2,52** (Ligue 1) a **2,74**(La Liga): uno scarto di un gol ogni cinque partite.Il dato interessante e' un altro. Serie A e Premier tirano di piu' — 26,3 e26,1 tiri a partita contro i 24,1 della Liga — ma producono **meno** xG. Sitira di piu' da posizioni peggiori.Ed e' esattamente il tipo di confronto che il piano iniziale non avrebbepotuto fare: prendeva tre leghe da tre stagioni distanti fino a otto anni, euno scarto del genere sarebbe stato indistinguibile dall'effetto dell'epoca.Qui la stagione e' la stessa per tutte e quattro.

---## 6. Il vantaggio di campo

In [12]:
gol_casa, gol_fuori = campionati["gol_casa"].sum(), campionati["gol_ospite"].sum()
xg_casa, xg_fuori = campionati["xg_casa"].sum(), campionati["xg_ospite"].sum()

print(f"gol   in casa {gol_casa:>6.0f}   fuori {gol_fuori:>6.0f}   →  {gol_casa / (gol_casa + gol_fuori):.1%} in casa")
print(f"xG    in casa {xg_casa:>6.0f}   fuori {xg_fuori:>6.0f}   →  {xg_casa / (xg_casa + xg_fuori):.1%} in casa")
print()
print(f"gol rispetto all'xG,  in casa: {(gol_casa / xg_casa - 1) * 100:+.1f}%")
print(f"gol rispetto all'xG,  fuori:   {(gol_fuori / xg_fuori - 1) * 100:+.1f}%")

gol   in casa   2284   fuori   1713   →  57.1% in casa
xG    in casa   2110   fuori   1637   →  56.3% in casa

gol rispetto all'xG,  in casa: +8.2%
gol rispetto all'xG,  fuori:   +4.7%


**Osservazione 6.** Il vantaggio di campo agisce **due volte**, e la seconda e'quella che non ti aspetti.Le squadre di casa producono piu' occasioni: il **56,3 %** dell'xG totale. Finqui e' noto. Ma segnano anche **piu' di quanto quelle occasioni valgano**:+8,2 % rispetto all'xG in casa, contro +4,7 % in trasferta.Non basta creare di piu': a parita' di qualita' dell'occasione, in casa siconverte meglio.

---## Le tre domande della dashboardDa queste osservazioni escono le tre domande a cui il progetto risponde, e cheordinano le nove viste.### 1. Quanto vale sapere dove sono i difensori?L'osservazione 3 mostra un fattore cinque nella conversione a distanza quasicostante. Due modelli addestrati sulle stesse partite — uno con le variabiliricavate dal fotogramma, uno senza — misurano quanto di quel segnale un modelloriesce davvero a catturare. **E' la domanda che il progetto esiste perrispondere.**### 2. Chi segna piu' di quanto dovrebbe, e per quanto tempo?La differenza fra gol e xG e' la misura piu' fraintesa del calcio analitico.Su un giocatore e mezza stagione e' quasi solo fortuna; su 43.849 tiri si puo'guardare quanto quello scarto persista. La vista Giocatori e la schedaindividuale servono a questo — con la soglia dei 500 minuti, perche' senza unasoglia il miglior marcatore per novanta minuti e' sempre uno entrato al 90°.### 3. Le leghe giocano davvero in modo diverso?L'osservazione 5 dice che si somigliano nei gol ma non nel modo di arrivarci:la Serie A tira di piu' e produce meno xG della Liga. Quattro campionati della**stessa stagione** permettono di dirlo senza che l'epoca faccia da variabilenascosta.